# Week 4 — Wednesday: Joining Tables — Keys and Relationships

**DATA 202 · Calvin University**

> But Ruth said, "Do not urge me to leave you or to return from following you. For where you go, I will go, and where you lodge, I will lodge. Your people shall be my people, and your God my God." — Ruth 1:16
>
> A join is a promise that two separate records are actually about the same thing. Ruth's declaration works because both sides commit to it — she binds herself to Naomi's people, completely, by name. Today you'll see what happens when only *one* side of that binding is on record: a plot that's been gardened all season with no paperwork to match, a registration with no garden behind it yet. `pd.merge()` doesn't know the difference between a relationship and a coincidence of matching text — it only knows whether the keys line up exactly.

Monday's whole class came back to one question, asked of a single table: **what is one row about?** Today we ask a harder version of it, asked of *two* tables at once: when one row in `yields` and one row in `registry` are each "about" some plot, how do you tell whether they're about the **same** plot? Answering that reliably is the entire idea behind a **key column** — and it's exactly what today is about.

Same community garden as Monday, but now **two tables**: the harvest data you already reshaped, and a second table — who's actually registered with the garden coordinator. They don't line up as neatly as you might expect.

**Today's plan (~50 min):**

| Time | Section |
|---|---|
| ~5 min | Load both tables, quick reshape recap |
| ~15 min | Part 1 — Keys and Relational Structure (SLO 04A) |
| ~25 min | Part 2 — The Four Join Types (SLO 04B) |
| ~5 min | Careful with Joining + what's next |

Same two stop-and-check cues as Monday: **🎯 Predict First** and **🙋 Quick Check**.

---
## Loading Both Tables

In [ ]:
import pandas as pd

yields = pd.read_csv("../../datasets/plot_yields.csv")
registry = pd.read_csv("../../datasets/gardener_registry.csv")

print(yields.shape, registry.shape)
registry.head()

`yields` — Monday's table — has 18 rows, one per harvested plot. `registry` also has 18 rows, but it's a **completely different subject**: one row per plot *registered* with the garden coordinator, with its own columns (`Garden_Site`, `Years_Gardening`, `Grows_Organic`) that `yields` has never heard of.

🙋 **Quick Check:** Ask Monday's question of each table separately. One row of `yields` is about ___. One row of `registry` is about ___. Those two answers sound similar ("a plot") — but is that actually specific enough to guarantee that row 5 of `yields` and row 5 of `registry` are about the *same* plot? What single fact would you need to check?

---
## Part 1: Keys and Relational Structure (SLO 04A) · ~15 min

Here's where "what is one row about?" leads: once two tables can each answer that question with something like *"one plot"*, the only way to actually connect a row in one to a row in the other is a column that names **which** plot, identically, in both places. That column is a **key**.

`yields` and `registry` share exactly one such column: `Plot_ID`.

* In `yields`, `Plot_ID` is a **primary key** — a column that uniquely identifies each row (no plot appears twice), and it's precisely the answer to "what is this row about."
* In `registry`, the very same `Plot_ID` column is also acting as a **foreign key** — a reference *pointing back* to a row that (we hope) exists in another table, asserting "this row is about that same plot, over there."

A key only does its job if the values match **exactly**, character for character — no fuzzier than the string-equality checks you've been writing all semester. "What is this row about?" is a question a human can answer with common sense; `pd.merge()` can only answer it by comparing text.

🎯 **Predict First:** Just by the fact that both tables have exactly 18 rows, do you expect *every* `Plot_ID` in `yields` to have a match in `registry`? Guess yes or no, then let's actually check.

In [ ]:
yields_ids = set(yields["Plot_ID"])
registry_ids = set(registry["Plot_ID"])

print("In yields but not registry:", sorted(yields_ids - registry_ids))
print("In registry but not yields:", sorted(registry_ids - yields_ids))

Two plots (`G17`, `G18`) were harvested all season on a verbal handshake with the coordinator — never formally registered. Two *other* plot numbers (`G19`, `G20`) are registered for **next** season and haven't been planted yet, so they have no harvest rows at all. 18 rows each, and yet only **16** plots are genuinely in both tables.

Every row in both tables would still answer "what is this row about?" with "a plot" — that hasn't changed. What's missing for `G17` and `G18` is a *partner row on the other side whose key value matches, character for character.* The key is exactly where that shared answer either holds up, row by row, or quietly breaks down.

---
### 🔨 Mini-Task A — Confirm It With `.isin()` (~4 min)

Without using set subtraction this time, use `.isin()` to build a boolean filter and confirm the same two results:

1. Filter `yields` to the rows whose `Plot_ID` is **not** in `registry["Plot_ID"]`. Assign to `yields_only`.
2. Filter `registry` to the rows whose `Plot_ID` is **not** in `yields["Plot_ID"]`. Assign to `registry_only`.

Do the `Plot_ID` values match what the set-based check above found?

In [ ]:
# Your code here


---
## Part 2: The Four Join Types (SLO 04B) · ~25 min

`pd.merge()` combines two tables on a shared key. `how=` decides what happens to a row that *doesn't* find a match on the other side:

| `how=` | Keeps | Unmatched rows get... |
|:---|:---|:---|
| `"inner"` | only rows matched in **both** tables | dropped completely, from both sides |
| `"left"` | every row from the **left** table | `NaN` filled in for the right table's columns |
| `"right"` | every row from the **right** table | `NaN` filled in for the left table's columns |
| `"outer"` | every row from **either** table | `NaN` filled in on whichever side is missing |

🎯 **Predict First:** Given what you just found — 16 plots in both tables, 2 (`G17`, `G18`) in `yields` only, 2 (`G19`, `G20`) in `registry` only — predict the row count for each join type below *before* running the cells.

In [ ]:
inner = pd.merge(yields, registry, on="Plot_ID", how="inner")
inner.shape

In [ ]:
left = pd.merge(yields, registry, on="Plot_ID", how="left")
left.shape

In [ ]:
right = pd.merge(yields, registry, on="Plot_ID", how="right")
right.shape

In [ ]:
outer = pd.merge(yields, registry, on="Plot_ID", how="outer")
outer.shape

16, 18, 18, 20 — four different row counts from the exact same two tables. Were your predictions right?

🙋 **Quick Check:** Which two plots vanish completely from `inner`? Which columns are `NaN` for `G17` and `G18` inside `left`? What about `G19` and `G20` inside `right`?

In [ ]:
left[left["Garden_Site"].isnull()][["Plot_ID", "Gardener"]]

In [ ]:
right[right["Week1_lbs"].isnull()][["Plot_ID", "Gardener_Name"]]

---
### 🔨 Mini-Task B — Pick the Right Join (~4 min)

The garden coordinator wants a report that includes **every plot that was actually harvested this season** — registered or not — with registry details filled in wherever available. Which single `how=` value gives them that in one `pd.merge()` call? Write the line below and assign it to `harvest_report`. Confirm its shape matches what you expect.

In [ ]:
# Your code here
harvest_report = None


---
### 🔨 Task — Organic vs. Non-Organic, Done Right (~8 min)

The coordinator's real question: **do organically-grown plots (`Grows_Organic == "Yes"`) harvest more, on average, than non-organic plots?**

Answering this correctly takes both of this week's skills, in order:

1. **Total each plot's season harvest.** Add a column `Total_lbs` to `yields` that sums `Week1_lbs` through `Week6_lbs` for each row.
2. **Join.** Merge `yields` and `registry` on `Plot_ID`, using whichever `how=` keeps every harvested plot even if it isn't registered (same choice as Mini-Task B). Call the result `merged`.
3. **Group and compare.** Group `merged` by `Grows_Organic` and compute the mean of `Total_lbs`. What happens to the plots with no `Grows_Organic` value at all — do they get silently dropped, or do they show up as their own group? Look closely at the output.

In [ ]:
# Your code here


---
## Careful with Joining

An `inner` join feels like the "safe," clean choice — no messy `NaN`s, every row fully filled in. But look at what it costs here: **`G17` and `G18` really were harvested this season** — real weeks, real pounds, sitting right there in `yields` the whole time. A report built only from `inner` would show a garden coordinator a season that's missing two entire plots' worth of real vegetables, with no error, no warning, nothing to flag that anything was left out.

That's not a pandas bug. It's a **choice** — `how="inner"` — made once, early, that quietly shapes every number computed after it. The plot that's really being harvested but isn't on anyone's official list is the easiest kind of contributor for a report to erase completely.

This week's reading picks up exactly here — tracing *why* a key mismatch like `G17` vs. nothing happens in the first place, and what it means that a join failure and a genuine absence of data can look identical from inside the merged table.

---
## Coming Up

| Topic | What's next |
|---|---|
| This week's reading | Follows the same kind of key mismatch through a *data journey* — who collected it, who cleaned it, who's still missing |
| Practice | Melting, pivoting, and joining together, on a new dataset |
| Week 5 | Clustering & Dimensionality Reduction — finding groups the data suggests, instead of ones we choose in advance |